In [1]:
# importing required modules
import matplotlib.pyplot as plt
from vasttools.query import Query
import pandas as pd

from vasttools.moc import VASTMOCS
from astropy import units as u

import numpy as np

from astropy.coordinates import SkyCoord

import scipy.stats as stats

In [2]:
import logging
logging.basicConfig(level=201)

In [3]:
#calculates a weighted mean flux density for a single source
def weighted_mean_flux_density(source_df):
    sum_1overvariance = 0
    for i in np.arange(source_df.shape[0]):
        if (not np.isnan(source_df['flux_peak'].iloc[i])):
            oneOverVariance = 1 / source_df['rms_image'].iloc[i]
            sum_1overvariance += (oneOverVariance ** 2)
    sumSovervariance = 0
    for i in np.arange(source_df.shape[0]):
        if (not np.isnan(source_df['flux_peak'].iloc[i])):
            S = source_df['flux_peak'].iloc[i]
            var = source_df['rms_image'].iloc[i] **2
            sumSovervariance += (S/var)
    if (sum_1overvariance == 0):
        return np.nan
    return sumSovervariance / sum_1overvariance       

In [4]:
#calculates chi_squared for a single source
def chi_squared(source_df):
    n_epochs = 0
    S_weighted = weighted_mean_flux_density(source_df)
    if (S_weighted == np.nan):
        return np.nan
    temp = 0
    for i in np.arange(source_df.shape[0]):
        if (not np.isnan(source_df['flux_peak'].iloc[i])):
            S = source_df['flux_peak'].iloc[i]
            var = source_df['rms_image'].iloc[i] ** 2
            num = (S - S_weighted)**2
            temp += num / var
            n_epochs += 1
    return (temp, n_epochs)

In [5]:
# make a query, and calculate a chi squared for the source found
def query_to_chi2(measurements_row):
    my_skycoord = SkyCoord(str(measurements_row.loc['ra_deg_cont']) + " " + str(measurements_row.loc['dec_deg_cont']), unit='deg')
    my_query = Query(coords=my_skycoord, epochs='all-vast', use_tiles=True, corrected_data=False)
    my_query.find_sources()
    print("        " + "0")
    return chi_squared(my_query.results[0].measurements[my_query.results[0].measurements['freq']==887.5])

In [6]:
# using query_to_chi2, calculate and input a chi2 and nu_detections value into a given source dataframe
def intelligent_chi2_from_rowdf(row_df):
    counter = 0
    for i in row_df.index.values:
        print("    " + str(i))
        row_df.loc[i, ['chi_squared', 'num_detections']] =  query_to_chi2(row_df.loc[i])
        if ((row_df.loc[i, 'chi_squared'] < 30)&(row_df.loc[i, 'num_detections'] > 40)):
            counter = counter + 1
        if (counter == 3):
            return "Completed"
    return "Completed"

In [7]:
psr_df = pd.read_csv('all_pulsar_measurements.csv')

/tmp/ipykernel_125/1176986871.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  psr_df = pd.read_csv('all_pulsar_measurements.csv')


In [8]:
psr_df.rename(columns={psr_df.columns[0] : 'JNAME'}, inplace=True)

In [27]:
psr_name = 'J1840-1207'

In [28]:
coords = psr_df[(psr_df['JNAME']==psr_name)&(psr_df['detection'])][['ra_deg_cont', 'dec_deg_cont']]
query_coord = SkyCoord(str(coords['ra_deg_cont'].iloc[0]) + " " + str(coords['dec_deg_cont'].iloc[0]), unit='deg')
new_query = Query(
    coords=query_coord, 
    epochs='23', 
    crossmatch_radius=3600., 
    search_around_coordinates=True, 
    use_tiles=True, 
    corrected_data=False
)
new_query.find_sources()
source_flux = psr_df[psr_df['JNAME']==psr_name]['flux_peak'].median()
new_results = new_query.results[(new_query.results['flux_peak']>(source_flux*0.8))&(new_query.results['distance']>15)]
new_results = new_results.reset_index()
for j in np.arange(new_results.shape[0]):
    if not (new_results.loc[j]['flux_int'] < (1.5 * new_results.loc[j]['flux_peak'])):
        new_results.drop(j, inplace=True)
new_results1 = new_results.sort_values('distance').reset_index(drop=True)

In [29]:
new_results1

,index,name,ra,dec,skycoord,stokes,fields,primary_field,epoch,field,...,spectral_index_err,spectral_curvature_err,rms_image,has_siblings,fit_is_estimate,spectral_index_from_TT,flag_c4,comment,detection,distance
0,2,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.260,0,0,1,0,,True,152.411700
1,0,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.253,0,0,1,0,,True,152.538787
2,4,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.270,0,0,1,0,,True,152.639157
3,5,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.265,0,0,1,0,,True,152.746877
4,3,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.298,0,0,1,0,,True,153.376522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1690,2,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.327,0,0,1,0,,True,3591.964423
1691,3,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.375,0,0,1,0,,True,3592.909986
1692,5,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.256,0,0,1,0,,True,3596.468922
1693,4,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.280,0,0,1,0,,True,3599.028389


In [30]:
data = new_results1
coords = SkyCoord(
    ra=data["ra_deg_cont"] * u.degree, dec=data["dec_deg_cont"] * u.degree
)

#gets rid of duplicate sources by looking at angular separation
checked_inds = []
for i in np.arange(coords.size):
    if i in checked_inds:
        continue
    for j in np.arange(coords.size):
        if j in checked_inds:
            continue
        if ((coords[i].separation(coords[j]).arcsec < 10) & (i != j)):
            checked_inds.append(j)
inds_to_delete = checked_inds
data.drop(inds_to_delete, inplace=True)

# Select bright sources (SNR >=8)
snr = data["flux_peak"] / data["rms_image"]

snr_mask = snr >= 8

#distance mask
#dist_mask = data['distance'] < 1800

# Filter the data
cut_data = data[
    (snr_mask) 
#    & (dist_mask)
]

In [31]:
cut_data

,index,name,ra,dec,skycoord,stokes,fields,primary_field,epoch,field,...,spectral_index_err,spectral_curvature_err,rms_image,has_siblings,fit_is_estimate,spectral_index_from_TT,flag_c4,comment,detection,distance
0,2,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.260,0,0,1,0,,True,152.411700
13,5,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.263,0,0,1,0,,True,310.309607
21,2,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.252,0,0,1,0,,True,328.252183
35,2,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.259,0,0,1,0,,True,464.052159
42,5,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.247,0,0,1,0,,True,508.160275
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1671,5,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.252,0,0,1,0,,True,3562.091043
1679,5,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.261,1,0,1,0,,True,3570.244753
1681,2,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.318,0,0,1,0,,True,3577.010886
1687,6,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],VAST_1831-12,23,VAST_1831-12,...,0.0,0.0,0.298,0,0,1,0,,True,3588.253308


In [32]:
cut_data.insert(0, 'chi_squared', np.nan)
cut_data.insert(0, 'num_detections', np.nan)

cut_data.reset_index(inplace=True)

In [33]:
intelligent_chi2_from_rowdf(cut_data)

    0
        0
    1
        0
    2
        0
    3
        0
    4
        0
    5
        0
    6
        0
    7
        0
    8
        0
    9
        0
    10
        0
    11
        0
    12
        0
    13
        0
    14
        0
    15
        0
    16
        0
    17
        0
    18
        0
    19
        0
    20
        0
    21
        0
    22
        0
    23
        0
    24
        0
    25
        0
    26
        0
    27
        0
    28
        0
    29
        0
    30
        0
    31
        0
    32
        0
    33
        0
    34
        0
    35
        0
    36
        0
    37
        0
    38
        0
    39
        0
    40
        0
    41
        0
    42
        0
    43
        0
    44
        0
    45
        0
    46
        0
    47
        0
    48
        0
    49
        0
    50
        0
    51
        0
    52
        0
    53
        0
    54
        0
    55
        0
    56
        0
    57
        0
    58
        0
    59


'Completed'

In [34]:
cut_data.sort_values(by=['chi_squared'])[cut_data['chi_squared'].notna()]

/tmp/ipykernel_125/3229756560.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cut_data.sort_values(by=['chi_squared'])[cut_data['chi_squared'].notna()]


,level_0,num_detections,chi_squared,index,name,ra,dec,skycoord,stokes,fields,...,spectral_index_err,spectral_curvature_err,rms_image,has_siblings,fit_is_estimate,spectral_index_from_TT,flag_c4,comment,detection,distance
10,82,3.0,1.872547e+00,4,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],...,0.0,0.0,0.281,1,0,1,0,,True,766.241301
27,199,23.0,1.542352e+01,0,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],...,0.0,0.0,0.239,0,0,1,0,,True,1181.041384
71,644,45.0,2.515761e+01,5,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],...,0.0,0.0,0.261,0,0,1,0,,True,2128.477688
68,618,41.0,2.898427e+01,5,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],...,0.0,0.0,0.263,0,0,1,0,,True,2035.346978
40,325,32.0,2.976168e+01,6,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],...,0.0,0.0,0.250,0,0,1,0,,True,1509.715781
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50,405,41.0,1.481528e+03,5,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],...,0.0,0.0,0.303,0,0,1,0,,True,1631.266425
31,242,47.0,3.038767e+03,5,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],...,0.0,0.0,0.307,0,0,1,0,,True,1301.845678
46,367,46.0,4.000896e+03,6,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],...,0.0,0.0,0.265,1,0,1,0,,True,1558.932764
19,138,47.0,1.030071e+05,5,source_184053.8-120735.1,280.22406,-12.12641,"<SkyCoord (ICRS): (ra, dec) in deg\n (280.2...",I,[VAST_1831-12],...,0.0,0.0,0.287,0,0,1,1,,True,1027.868308


In [35]:
cut_data.sort_values(by=['chi_squared'])[cut_data['chi_squared'].notna()].to_csv('manual_ctrl_lists/' + psr_name + '.csv')

/tmp/ipykernel_125/1413666567.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cut_data.sort_values(by=['chi_squared'])[cut_data['chi_squared'].notna()].to_csv('manual_ctrl_lists/' + psr_name + '.csv')
